<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# IMU Estimation and Drift

The accelerometer and gyroscope introduced in [Notebook 1](./1-imu-basics.ipynb) provide measurements from which the Duckiedrone's orientation and motion can be estimated.

This notebook addresses the following questions:

- How can a stationary proper-acceleration measurement help determine the Duckiedrone's roll and pitch, and why does it not provide an absolute yaw reference?

- What is required to recover position from proper acceleration, and how do measurement errors affect that calculation?

- What does subtracting a stationary average correct, and why does a longer averaging window not guarantee less position drift?

This notebook examines the references available for orientation, then uses a controlled model and a recorded signal to investigate how measurement errors accumulate when estimating position.

<span id="why-the-imu-data-matters"></span>

## Why the IMU?

The flight controller uses an estimate of orientation, or attitude, together with measured angular velocity to control the Duckiedrone's rotation toward a desired attitude. Neither sensing element measures orientation directly: the gyroscope measures how orientation changes, while the accelerometer supplies a reference for tilt when linear acceleration is negligible.

An absolute reference relates a quantity to an independently defined direction or location. For example, gravity defines the local vertical, while satellite positioning (GPS) can relate a location to Earth-fixed coordinates. A relative estimate instead describes a change from a starting state, such as a rotation from the initial heading. An absolute reference is useful to constrain accumulated error, although its own measurements still have uncertainty.

For a stationary Duckiedrone, support against gravity produces an upward proper acceleration of approximately $9.81\,\mathrm{m/s^2}$. Tilting the Duckiedrone changes how this vector projects onto its axes, so its components provide information about roll and pitch. During accelerated motion, however, the reading also contains the contribution from linear acceleration; interpreting it entirely as a tilt reference would give an incorrect orientation.

Rotating about the local vertical leaves the stationary proper-acceleration vector unchanged. Consequently, that vector cannot distinguish headings and supplies no absolute yaw reference. Gyroscope integration can track a heading change from an initial value, but rate errors accumulate unless another source constrains the estimate. A magnetometer can provide heading information by measuring the Earth's magnetic field, subject to calibration and local magnetic disturbances.

## What an IMU does not measure

An accelerometer and gyroscope do not directly report position or linear velocity of the sensor or the Duckiedrone. Recovering those quantities requires an initial position and velocity, an orientation estimate, and a calculation that accounts for gravity. The Duckiedrone's flight controller runs an estimation algorithm called Extended Kalman Filter 2 (EKF2). [PX4's EKF2 documentation](https://docs.px4.io/main/en/advanced_config/tuning_the_ecl_ekf.html) describes how it uses IMU data to predict motion and combines that prediction with other available measurements. Recursive estimators are beyond the scope of this LX, but [useful state estimation Duckietown resources](https://docs.duckietown.com/ente/duckietown-manual/80-instructor-manual/available-resources/slides/estimation/00-estimation-overview.html) are available in the Instructor Manual for the curious learner.

### Recovering motion from proper acceleration

The sensor frame $\mathrm{S}$ moves with the sensor, the body frame $\mathrm{B}$ moves with the Duckiedrone, and the world frame $\mathrm{W}$ is treated as stationary. Let $\mathbf{R}_{\mathrm{BS}}$ be the fixed rotation matrix that converts sensor-frame components into body-frame components:

$$
\mathbf{f}_{\mathrm{B}}(t)
= \mathbf{R}_{\mathrm{BS}}\,\mathbf{f}_{\mathrm{S}}(t),
\qquad
\tilde{\mathbf{f}}_{\mathrm{B}}(t)
= \mathbf{R}_{\mathrm{BS}}\,\tilde{\mathbf{f}}_{\mathrm{S}}(t).
$$

When the sensor and body axes align, $\mathbf{R}_{\mathrm{BS}}=\mathbf{I}$, where $\mathbf{I}$ is the three-by-three identity matrix. Let $\mathbf{R}_{\mathrm{WB}}(t)$ convert body-frame components into world-frame components. Writing linear and gravitational acceleration in the world frame as $\mathbf{a}_{\mathrm{W}}(t)$ and $\mathbf{g}_{\mathrm{W}}$, [Equation (2) in Notebook 1](./1-imu-basics.ipynb#equation-2) becomes:

$$
\mathbf{a}_{\mathrm{W}}(t)
= \mathbf{R}_{\mathrm{WB}}(t)\,\mathbf{R}_{\mathrm{BS}}\,\mathbf{f}_{\mathrm{S}}(t)
+ \mathbf{g}_{\mathrm{W}}.
$$

Using measured proper acceleration gives an approximation:

<span id="equation-1"></span>

$$
\mathbf{a}_{\mathrm{W}}(t)
\approx
\mathbf{R}_{\mathrm{WB}}(t)\,\mathbf{R}_{\mathrm{BS}}\,\tilde{\mathbf{f}}_{\mathrm{S}}(t)
+ \mathbf{g}_{\mathrm{W}}.
\tag{1}
$$

[Equation (1)](#equation-1) first rotates the measurement from sensor to body axes, then from body to world axes, before adding gravitational acceleration in the world frame. The approximation assumes known rotations and small or corrected sensor errors. [Notebook 3](./3-rotation-representations.ipynb) develops these rotations.

Let $\mathbf{x}_{\mathrm{W}}(0)$ and $\mathbf{v}_{\mathrm{W}}(0)$ be the initial world-frame position and velocity, and let $\mathbf{x}_{\mathrm{W}}(t)$ be position at time $t$. Integrating world-frame linear acceleration twice gives:

<span id="equation-2"></span>

$$
\mathbf{x}_{\mathrm{W}}(t)
= \mathbf{x}_{\mathrm{W}}(0)
+ \mathbf{v}_{\mathrm{W}}(0)t
+ \int_0^t \int_0^\tau \mathbf{a}_{\mathrm{W}}(s)\,\mathrm{d}s\,\mathrm{d}\tau.
\tag{2}
$$

Here, $s$ and $\tau$ are integration times. Updating a position estimate with [Equation (2)](#equation-2) from motion measurements and a starting state is known as [dead reckoning](https://en.wikipedia.org/wiki/Dead_reckoning).

Dead reckoning can bridge the intervals between external corrections, but its errors accumulate while no independent reference constrains the result. The measurement model in [Equation (3) in Notebook 1](./1-imu-basics.ipynb#equation-3) separates ideal proper acceleration from an accelerometer bias and random measurement noise. Both affect the integrated estimate, so the first step is to examine how a stationary signal varies.

### Noise and bias in measurements

The [sample message in Notebook 1](./1-imu-basics.ipynb#gathering-imu-measurements-from-a-duckiedrone) provides a single observation, but a sequence is needed to describe variation over time.

The supplied [imu_still_5min.csv recording](../assets/data/imu_still_5min.csv) contains timestamps and three proper-acceleration components, named `t`, `ax`, `ay`, and `az`, collected from a stationary, level Duckiedrone with its motors off.

[Table 1](#table-1) summarizes the first 1,500 rows, spanning approximately $30\,\mathrm{s}$. Each standard deviation is calculated about that component's sample mean using the sample convention, with denominator $N-1$ for $N$ samples. The code in [Activity 2](#activity-2) reproduces these statistics.

<a id="table-1"></a>

Table 1: Statistics from the first 1,500 samples of the supplied recording, spanning approximately $30\,\mathrm{s}$. Standard deviations use the sample convention.

| Quantity | Units | Mean | Standard deviation | Minimum | Maximum |
| --- | --- | --- | --- | --- | --- |
| $\tilde{f}_x(t_k)$ | $\mathrm{m/s^2}$ | $-0.069$ | $0.014$ | $-0.114$ | $-0.027$ |
| $\tilde{f}_y(t_k)$ | $\mathrm{m/s^2}$ | $-0.096$ | $0.018$ | $-0.141$ | $-0.044$ |
| $\tilde{f}_z(t_k)$ | $\mathrm{m/s^2}$ | $+9.809$ | $0.141$ | $+9.566$ | $+10.043$ |

The spread of the readings describes variation during this interval. While averaging can reduce the contribution of zero-mean random noise, it is still a finite average with uncertainty. The horizontal means also differ from zero. A small tilt projects gravity onto those axes, while sensor bias can contribute an additional offset. A single stationary orientation does not separate these effects, so the horizontal means alone do not establish accelerometer bias.

The CSV contains no gyroscope channels or frame identifier, so its column names alone do not establish the chip-to-body mapping. The next experiment uses `az` as an approximately vertical signal under a fixed-orientation assumption.

<span id="why-the-bias-is-fatal-to-dead-reckoning"></span>

### The effect of bias in position estimates: why "IMUs drift"

Consider one axis whose orientation is fixed and for which gravity has been accounted for correctly. Let $b$ be a constant residual acceleration error in $\mathrm{m/s^2}$, $t$ the elapsed time in seconds, and $\varepsilon(t)$ the resulting position error in meters. With zero initial position and velocity error, integrating $b$ once produces a velocity error $bt$, and integrating again gives:

<span id="equation-3"></span>

$$
\varepsilon(t)=\tfrac{1}{2}bt^2.
\tag{3}
$$

The position error grows quadratically because the erroneous velocity itself continues to increase. Without an independent observation, this calculation cannot distinguish a constant measurement offset from an actual constant acceleration.

Let $g_0=9.80665\,\mathrm{m/s^2}$ denote standard gravity. Accelerometer offsets are often expressed in milligravity units, written here as $\mathrm{m}g_0$, where $1\,\mathrm{m}g_0=10^{-3}g_0$. For an illustrative residual error of $1\,\mathrm{m}g_0$, Equation (3) gives:

$$
\varepsilon(60\,\mathrm{s})
=\tfrac{1}{2}(0.00980665\,\mathrm{m/s^2})(60\,\mathrm{s})^2
\approx 17.7\,\mathrm{m}.
$$

This is a consequence of the assumed residual error and integration time, rather than a specification of the Duckiedrone's sensor. Note the quadratic dependence on time: doubling time multiplies the error by four, whereas reducing the bias by a factor of five reduces the error by the same factor.

### Activity 1: an ideal Duckiedrone with one flaw

<a id="activity-1"></a>

In this controlled model, a Duckiedrone remains on the floor of a room with a $2.5\,\mathrm{m}$ ceiling. Its orientation and gravity correction are exact, its initial velocity and altitude are zero, and the only error is a constant vertical accelerometer bias $b$.

The green line marks the true altitude; the red points show the altitude calculated from [Equation (3)](#equation-3).

1. Keep the bias at $5\,\mathrm{m}g_0$ and compare integration durations of $10$ and $20\,\mathrm{s}$. Predict the change in position error before moving the time slider.

2. Keep the duration at $20\,\mathrm{s}$ and reduce the bias to $1\,\mathrm{m}g_0$. Compare the reduction in error with the reduction in bias.

3. Restore the initial settings of $5\,\mathrm{m}g_0$ and $20\,\mathrm{s}$.

The plot remains focused on the room, so estimates above its displayed range disappear from view. The title and printed value report the final error even when it lies outside the plot. The illustrative $50\,\mathrm{Hz}$ time grid is chosen for this model; it is not a measured chip sampling rate.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import FloatSlider, interact

# Variables use the notation introduced above:
#   b     Vertical accelerometer bias [m/s^2].
#   t_k   Discrete sample times [s].
#   x     Dead-reckoned altitude [m].

G = 9.80665  # Standard-gravity magnitude [m/s^2].
SAMPLE_RATE = 50.0  # Illustrative IMU sample rate [Hz].
TRUE_ALT = 0.0  # Ground-truth altitude [m].
CEILING = 2.5  # Room height [m].


def ideal_drift(bias_mg=5.0, duration_s=20.0):
    """Compare where the Duckiedrone really is with its dead-reckoned estimate."""
    b = bias_mg * G / 1000.0  # Convert milli-g to m/s^2.

    # An IMU reports at discrete instants, so build the sample times explicitly.
    t_k = np.linspace(0.0, duration_s, int(duration_s * SAMPLE_RATE) + 1)

    # A constant bias gives the closed-form double integral x = (1/2) b t^2.
    x = TRUE_ALT + 0.5 * b * t_k**2

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.axhspan(
        CEILING, 10, color="0.92"
    )  # Shade the region above the ceiling.
    ax.axhline(CEILING, color="0.5", lw=1)
    ax.text(
        duration_s * 0.99,
        CEILING + 0.06,
        "ceiling",
        ha="right",
        fontsize=9,
        color="0.4",
    )

    # Ground-truth altitude.
    ax.axhline(TRUE_ALT, color="tab:green", lw=2.5, label="true altitude")

    # Plot a subset of discrete estimates so their markers remain distinguishable.
    stride = max(1, len(t_k) // 120)
    ax.plot(
        t_k[::stride],
        x[::stride],
        color="tab:red",
        linestyle="none",
        marker=".",
        markersize=6,
        label="dead-reckoned altitude (display subsampled)",
    )

    ax.set_ylim(-0.15, CEILING + 0.5)
    ax.set_yticks(np.arange(0.0, CEILING + 0.5, 0.5))
    ax.set_xlim(0, duration_s)
    ax.set_xlabel("time [s]")
    ax.set_ylabel("altitude [m]")
    ax.legend(loc="center right")

    ax.set_title(
        f"bias b = {bias_mg:.1f} mg = {b:.3f} m/s^2    "
        f"after {t_k[-1]:.0f} s it is off by {x[-1] - TRUE_ALT:.1f} m"
    )
    print(f"At {t_k[-1]:.0f} s: position error = {x[-1] - TRUE_ALT:+.3f} m")  # noqa: T201
    fig.tight_layout()
    plt.show()


interact(
    ideal_drift,
    bias_mg=FloatSlider(
        value=5.0,
        min=0.1,
        max=5.0,
        step=0.1,
        description="bias [mg]",
        style={"description_width": "110px"},
        continuous_update=False,
    ),
    duration_s=FloatSlider(
        value=20.0,
        min=5.0,
        max=300.0,
        step=1.0,
        description="time [s]",
        style={"description_width": "110px"},
        continuous_update=False,
    ),
);

### Activity 2: the same experiment on a physical Duckiedrone

<a id="activity-2"></a>

The supplied [CSV recording](../assets/data/imu_still_5min.csv) contains measurement samples spanning about $300\,\mathrm{s}$ at about 50 samples per second, of a stationary Duckiedrone resting on the floor with its motors off. Under those conditions, the true velocity and change in altitude are zero.

This experiment treats the recorded $z$ axis as vertical and holds its orientation fixed. It subtracts an initial stationary average, then integrates the remaining signal twice. The average includes the gravity projection and sensor offset in that pose, so this procedure establishes a reference for the experiment rather than performing a full accelerometer calibration.

Let $t_k$ be the timestamp of sample $k$, measured from the beginning of the recording, and let $\tilde{f}_{z,k}$ denote its recorded proper-acceleration component in $\mathrm{m/s^2}$. The subscript $z$ refers to the CSV column used here. The elapsed time between samples is $\Delta t_k=t_k-t_{k-1}$.

1. Select a reference duration $T_{\mathrm{ref}}$ and average the $N$ samples for which $t_k<T_{\mathrm{ref}}$:

   $$
   \hat{r}=\frac{1}{N}\sum_{k=0}^{N-1}\tilde{f}_{z,k}.
   $$

   The reference $\hat{r}$ has units of $\mathrm{m/s^2}$ and includes finite-sample noise as well as the stationary signal.

2. For subsequent samples, subtract that reference to form an acceleration estimate:

   $$
   \hat{a}_{z,k}=\tilde{f}_{z,k}-\hat{r}.
   $$

   Ideally this is zero while the Duckiedrone remains stationary in the same orientation. For general motion, the full vector must instead be corrected for measurement error, rotated into the world frame, and combined with gravity, as described in [Equation (1)](#equation-1).

3. Start integration at the first subsequent sample, indexed $N$, with estimated vertical velocity $v_N=0$ and altitude $x_N=0$. For $k\geq N+1$, use the trapezoidal rule, which averages the values at the two ends of each time interval:

<span id="equation-4"></span>

$$
v_k=v_{k-1}
+\tfrac{1}{2}\left(\hat{a}_{z,k-1}+\hat{a}_{z,k}\right)\Delta t_k.
\tag{4}
$$

<span id="equation-5"></span>

$$
x_k=x_{k-1}
+\tfrac{1}{2}\left(v_{k-1}+v_k\right)\Delta t_k.
\tag{5}
$$

Velocity is in $\mathrm{m/s}$ and altitude in $\mathrm{m}$. The code implements [Equation (4)](#equation-4) and [Equation (5)](#equation-5) using the recorded timestamp differences rather than assuming every interval is exactly $0.02\,\mathrm{s}$.

The first slider selects the reference duration and the second selects the subsequent integration duration. Increasing the first duration changes both the reference estimate and the segment being integrated, so this comparison does not isolate the effect of averaging more samples.

1. Start with a $10\,\mathrm{s}$ reference and a $60\,\mathrm{s}$ integration window. Compare the estimated altitude with the stationary ground truth.

2. Keep the integration window at $60\,\mathrm{s}$ and vary the reference duration. Observe whether the final error decreases consistently.

3. Set the reference duration to $120\,\mathrm{s}$ and the integration duration to $180\,\mathrm{s}$. Interpret the sign of the final altitude error, then restore the initial settings.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt  # noqa: F811
import numpy as np  # noqa: F811
from ipywidgets import FloatSlider, interact

# Variables use the notation from the procedure above:
#   f_z_tilde_all  Recorded vertical proper-acceleration samples [m/s^2].
#   t_all          Recording times [s].
#   r_hat          Stationary reference [m/s^2].
#   a_z_hat        Baseline-corrected vertical acceleration [m/s^2].
#   t_k            Elapsed time since integration began [s].
#   v, x           Dead-reckoned velocity [m/s] and altitude [m].

CEILING = 2.5  # Room height [m].
FLOOR = 0.0  # Ground-truth altitude [m].

# Load five minutes of vertical proper-acceleration data from a stationary Duckiedrone.
# Columns: t [s], ax, ay, az [m/s^2].
data_path = Path("assets/data/imu_still_5min.csv")
if not data_path.exists():
    data_path = Path("../assets/data/imu_still_5min.csv")
data = np.genfromtxt(data_path, delimiter=",", names=True)
t_all = data["t"] - data["t"][0]
f_z_tilde_all = data["az"]

# Reproduce Table 1 using sample standard deviation (ddof=1).
print("First 1500 rows: mean, sample standard deviation, minimum, maximum [m/s^2]")  # noqa: T201
for column in ("ax", "ay", "az"):
    values = data[column][:1500]
    print(  # noqa: T201
        f"{column}: {values.mean():+.3f}, {values.std(ddof=1):.3f}, "
        f"{values.min():+.3f}, {values.max():+.3f}"
    )


def real_drift(reference_s=10.0, watch_s=60.0):
    """Dead reckon altitude from the recording, after estimating a stationary reference."""
    # The mean includes sensor bias and the proper acceleration produced by
    # supporting the stationary Duckiedrone against gravitational acceleration.
    r_hat = f_z_tilde_all[t_all < reference_s].mean()

    # Select post-reference samples and measure time from the first one.
    integrating = (t_all >= reference_s) & (t_all <= reference_s + watch_s)
    sample_times = t_all[integrating]
    t_k = sample_times - sample_times[0]

    # Remove the stationary reference; the result is ideally zero while level and still.
    a_z_hat = f_z_tilde_all[integrating] - r_hat

    # Approximate the two integrals with the trapezoidal rule.
    dt = np.diff(t_k)
    v = np.concatenate(
        ([0.0], np.cumsum(0.5 * (a_z_hat[:-1] + a_z_hat[1:]) * dt))
    )
    x = FLOOR + np.concatenate(([0.0], np.cumsum(0.5 * (v[:-1] + v[1:]) * dt)))

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.axhspan(CEILING, 10, color="0.92")
    ax.axhspan(-10, FLOOR, color="0.88")
    ax.axhline(CEILING, color="0.5", lw=1)
    ax.text(
        t_k[-1] * 0.99,
        CEILING + 0.06,
        "ceiling",
        ha="right",
        fontsize=9,
        color="0.4",
    )

    # Ground-truth altitude.
    ax.axhline(FLOOR, color="tab:green", lw=2.5, label="true altitude")

    # Plot a subset of discrete estimates so their markers remain distinguishable.
    stride = max(1, len(t_k) // 120)
    ax.plot(
        t_k[::stride],
        x[::stride],
        color="tab:red",
        linestyle="none",
        marker=".",
        markersize=6,
        label="dead-reckoned altitude (display subsampled)",
    )

    ax.set_ylim(-1.0, CEILING + 0.5)
    ax.set_xlim(0, t_k[-1])
    ax.set_xlabel("time since integration started [s]")
    ax.set_ylabel("altitude [m]")
    ax.legend(loc="upper right")

    n_reference = len(t_all[t_all < reference_s])
    ax.set_title(
        f"reference: {reference_s:.0f} s, {n_reference} samples    "
        f"after {t_k[-1]:.0f} s of dead reckoning it is off by {x[-1] - FLOOR:+.1f} m"
    )
    print(  # noqa: T201
        f"Stationary reference = {r_hat:.6f} m/s^2; "
        f"final velocity = {v[-1]:+.3f} m/s; "
        f"altitude error = {x[-1] - FLOOR:+.3f} m"
    )
    fig.tight_layout()
    plt.show()


interact(
    real_drift,
    reference_s=FloatSlider(
        value=10.0,
        min=1.0,
        max=120.0,
        step=1.0,
        description="reference [s]",
        style={"description_width": "110px"},
        continuous_update=False,
    ),
    watch_s=FloatSlider(
        value=60.0,
        min=10.0,
        max=180.0,
        step=5.0,
        description="watch [s]",
        style={"description_width": "110px"},
        continuous_update=False,
    ),
);

### What the model and the recording show

Subtracting a stationary reference leaves small residual accelerations that the integration interprets as motion. In the supplied recording, the resulting position estimate can move far from the floor even though the Duckiedrone remained stationary. The sign of that error also matters: a negative altitude represents apparent motion below the floor, not a value to discard from the calculation.

A longer reference window changes the average, but it also changes the later samples used for integration. The final error therefore need not decrease as the reference duration increases. A small mean error over the complete recording can coexist with differences between shorter intervals, and those differences accumulate when integrated twice.

These experiments show why a stationary average alone cannot provide reliable position recovery. Independent observations can constrain accumulated error in an aided estimator, whereas the calculation here has only the accelerometer signal and its initial conditions.

## Summary and next steps

A stationary proper-acceleration vector provides a vertical reference that constrains roll and pitch. Gravity gives no yaw reference, so yaw from the gyroscope drifts unless something else corrects it. Position recovery also requires orientation, gravity compensation, and initial conditions; residual acceleration errors then accumulate through integration.

The fixed-orientation experiments expose that accumulation without requiring changing coordinate frames. For a rotating Duckiedrone, the same physical vector has different components in different frames. [Notebook 3](./3-rotation-representations.ipynb) develops the rotations needed to relate those components.

## Further reading

[Duckiedrone sensor calibration](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/50-sensor-calibrations.html) provides the platform's calibration procedure, which is distinct from the stationary reference subtraction used here.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()